In [4]:
# import cv2
# import numpy as np
# import pyautogui
# import time
# import HandTrackingModule as htm
# import math

# wCam, hCam = 1280, 720
# frameR = 100
# smoothening = 5
# pLocX, pLocY = 0, 0
# cLocX, cLocY = 0, 0
# dragging = False

# cap = cv2.VideoCapture(0)
# cap.set(3, wCam)
# cap.set(4, hCam)

# detector = htm.HandDetector(maxHands=2)
# screenW, screenH = pyautogui.size()

# while True:
#     success, img = cap.read()
#     img = detector.findHands(img)
#     lmList = detector.findPosition(img)

#     if len(lmList) != 0:
#         x1, y1 = lmList[8][1:]  # Index finger
#         x2, y2 = lmList[4][1:]  # Thumb
#         x3, y3 = lmList[12][1:]  # Middle finger

#         fingers = detector.fingersUp()

#         # Move Mode - Only index finger up
#         if fingers == [0, 1, 0, 0, 0]:
#             x3 = np.interp(x1, (frameR, wCam - frameR), (0, screenW))
#             y3 = np.interp(y1, (frameR, hCam - frameR), (0, screenH))
#             cLocX = pLocX + (x3 - pLocX) / smoothening
#             cLocY = pLocY + (y3 - pLocY) / smoothening
#             pyautogui.moveTo(cLocX, cLocY)
#             pLocX, pLocY = cLocX, cLocY
#             dragging = False

#         # Dragging - Index and middle finger down
#         elif fingers[1] == 1 and fingers[2] == 1:
#             if not dragging:
#                 pyautogui.mouseDown()
#                 dragging = True
#             x3 = np.interp(x1, (frameR, wCam - frameR), (0, screenW))
#             y3 = np.interp(y1, (frameR, hCam - frameR), (0, screenH))
#             pyautogui.moveTo(x3, y3)

#         elif dragging:
#             pyautogui.mouseUp()
#             dragging = False

#         # Left Click - Thumb and index close
#         if fingers[0] == 1 and fingers[1] == 1 and fingers[2] == 0:
#             length = math.hypot(x2 - x1, y2 - y1)
#             if length < 40:
#                 pyautogui.click()
#                 time.sleep(0.3)

#         # Right Click - Index and middle fingers close
#         if fingers[1] == 1 and fingers[2] == 1:
#             length = math.hypot(x1 - x3, y1 - y3)
#             if length < 40:
#                 pyautogui.rightClick()
#                 time.sleep(0.3)

#         # Scroll
#         if fingers[1] == 1 and fingers[2] == 1 and fingers[3] == 0:
#             if y1 < hCam // 2:
#                 pyautogui.scroll(20)
#             else:
#                 pyautogui.scroll(-20)

#         # Zoom (using both hands)
#         if detector.results.multi_hand_landmarks and len(detector.results.multi_hand_landmarks) == 2:
#             lmList1 = detector.findPosition(img, handNo=0, draw=False)
#             lmList2 = detector.findPosition(img, handNo=1, draw=False)
#             if len(lmList1) > 8 and len(lmList2) > 8:
#                 x1, y1 = lmList1[8][1:]
#                 x2, y2 = lmList2[8][1:]
#                 dist = math.hypot(x2 - x1, y2 - y1)
#                 if dist > 300:
#                     pyautogui.hotkey('ctrl', '+')  # Zoom in
#                     time.sleep(0.3)
#                 elif dist < 100:
#                     pyautogui.hotkey('ctrl', '-')  # Zoom out
#                     time.sleep(0.3)

#     cv2.imshow("Virtual Mouse", img)
#     if cv2.waitKey(1) & 0xFF == ord('q'):
#         break

# cap.release()
# cv2.destroyAllWindows()


In [ ]:
import cv2
import numpy as np
import pyautogui
import time
import HandTrackingModule as htm
import math

wCam, hCam = 1280, 720  # Increased resolution
frameR = 100  # Frame Reduction for mouse movement area
smoothening = 5  # Lower = faster

pLocX, pLocY = 0, 0
cLocX, cLocY = 0, 0
drag = False
zoom_start_distance = 0
zoom_threshold = 40  # Minimum distance change to trigger zoom

cap = cv2.VideoCapture(0)
cap.set(3, wCam)
cap.set(4, hCam)

detector = htm.HandDetector(maxHands=1, detectionCon=0.8)
screenW, screenH = pyautogui.size()

while True:
    success, img = cap.read()
    img = detector.findHands(img)
    lmList = detector.findPosition(img, draw=False)

    if len(lmList) != 0:
        x1, y1 = lmList[8][1:]  # Index finger
        x2, y2 = lmList[4][1:]  # Thumb
        x3, y3 = lmList[12][1:]  # Middle finger

        fingers = detector.fingersUp()

        # Moving Mode - Only Index Finger Up
        if fingers[1] == 1 and fingers[2] == 0:
            x3 = np.interp(x1, (frameR, wCam - frameR), (0, screenW))
            y3 = np.interp(y1, (frameR, hCam - frameR), (0, screenH))

            cLocX = pLocX + (x3 - pLocX) / smoothening
            cLocY = pLocY + (y3 - pLocY) / smoothening

            pyautogui.moveTo(cLocX, cLocY)
            pLocX, pLocY = cLocX, cLocY

        # Left Click - Thumb and Index Finger Close
        if fingers[0] == 1 and fingers[1] == 1:
            length = math.hypot(x2 - x1, y2 - y1)
            if length < 40:
                pyautogui.click()
                time.sleep(0.3)

        # Right Click - Index and Middle Finger Together
        if fingers[1] == 1 and fingers[2] == 1 and fingers[3] == 0:
            length = math.hypot(x1 - x3, y1 - y3)
            if length < 40:
                pyautogui.rightClick()
                time.sleep(0.3)

        # Drag - Only Index and Middle Finger Up
        if fingers[1] == 1 and fingers[2] == 1 and fingers[3] == 0 and fingers[0] == 0:
            if not drag:
                pyautogui.mouseDown()
                drag = True
            x3 = np.interp(x1, (frameR, wCam - frameR), (0, screenW))
            y3 = np.interp(y1, (frameR, hCam - frameR), (0, screenH))
            pyautogui.moveTo(x3, y3)
        else:
            if drag:
                pyautogui.mouseUp()
                drag = False

        # Zoom Gesture - Thumb and Index far apart
        if fingers[0] == 1 and fingers[1] == 1 and fingers[2] == 0:
            distance = math.hypot(x2 - x1, y2 - y1)
            if zoom_start_distance == 0:
                zoom_start_distance = distance
            else:
                diff = distance - zoom_start_distance
                if abs(diff) > zoom_threshold:
                    if diff > 0:
                        pyautogui.hotkey('ctrl', '+')
                    else:
                        pyautogui.hotkey('ctrl', '-')
                    zoom_start_distance = distance
                    time.sleep(0.4)
        else:
            zoom_start_distance = 0

    cv2.imshow("Virtual Mouse", img)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
